# Konwersja z json na csv

In [ ]:
import json
import sys
from pathlib import Path


def get_property(obj, name, default=False):
    for p in obj.get("properties", []):
        if p["name"] == name:
            return int(p["value"])
    return int(default)


def convert(input_file, output_file):

    with open(input_file, "r", encoding="utf-8") as f:
        tiled = json.load(f)


    width = tiled["width"]
    height = tiled["height"]

    tile_w = tiled["tilewidth"]
    tile_h = tiled["tileheight"]


    tile_layer = None
    object_layer = None


    for layer in tiled["layers"]:
        if layer["type"] == "tilelayer":
            tile_layer = layer

        if layer["type"] == "objectgroup":
            object_layer = layer


    with open(output_file, "w", encoding="utf-8") as out:

        out.write("// Generated from Tiled\n\n")
        
        out.write("#pragma once\n\n")
        out.write("#include <stdint.h>\n\n")


        #
        # MAP
        #
        if tile_layer:

            out.write(f"#define MAP_WIDTH {width}\n")
            out.write(f"#define MAP_HEIGHT {height}\n\n")

            out.write(
                "const uint16_t map_tiles[] = {\n"
            )

            data = tile_layer["data"]

            for y in range(height):
                out.write("    ")

                row = data[y*width:(y+1)*width]

                out.write(",".join(map(str,row)))

                out.write(",\n")


            out.write("};\n\n")


        #
        # OBJECTS
        #
        if object_layer:

            out.write("""
typedef struct
{
    uint16_t x;
    uint16_t y;

    uint16_t type;

    uint8_t coll_up;
    uint8_t coll_down;
    uint8_t coll_left;
    uint8_t coll_right;

} MapObject;


""")


            objects = object_layer["objects"]


            out.write(
                f"const MapObject map_objects[{len(objects)}] = {{\n"
            )


            for obj in objects:

                # Tiled zapisuje w pikselach
                x = int(obj["x"] // tile_w)
                y = int(obj["y"] // tile_h)


                typ = obj.get("id",0)


                up = get_property(obj,"coll_up")
                down = get_property(obj,"coll_down")
                left = get_property(obj,"coll_left")
                right = get_property(obj,"coll_right")


                out.write(
                    f"    {{{x},{y},{typ},"
                    f"{up},{down},{left},{right}}},\n"
                )


            out.write("};\n")


    print("Generated:", output_file)



if __name__ == "__main__":

    if len(sys.argv) != 3:
        print(
            "Usage:\n"
            " python tiled_to_c.py map.json map.h"
        )
        sys.exit(1)


    convert(
        sys.argv[1],
        sys.argv[2]
    )

JSONDecodeError: Expecting property name enclosed in double quotes: line 3 column 9 (char 15)